In [1]:
!pip install torchvision

In [2]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [3]:
# Image load => transform => dataset of all imgs
class ImageProcessor:
    def __init__(self, root_dir_path, transformations):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of path for all images
        self.all_imgs_path = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_imgs_path)

    def __getItem__(self, idx):
        img_path = self.all_imgs_path[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)
        return img
        
        
    

In [4]:
root_dir_path = "./img_align_celeba"
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178*218 => 178*178
    transforms.Resize(64), # 64*64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # [-1, 1]
])

In [5]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [6]:
dataloader = DataLoader(dataset, batch_size = 128, shuffle = True)
print(type(dataloader))

<class 'torch.utils.data.dataloader.DataLoader'>


# Generator Network

In [8]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [12]:
class Generator(nn.Module): # Generator gets fake data which we represent here by z which is basically the random vecotrs of noise(fake imgs data)
    def __init__(self, z_dim = 100, img_channels = 3): # 3 is for RGB, z is noise the image random vectors
        super(Generator, self).__init__()

        # Fully connected layer

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100 => 256
            nn.ReLU(),


            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 64*64*img_channels),
            nn.Tanh() # [-1, 1] normalize pixel in this range

        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img


        # fake image = 64 * 64 * 3 * batchSize so thats's 4D

# Discriminator Network

In [13]:
class Discriminator(nn.Module): 
    def __init__(self, img_channels = 3): # 3 is for RGB,
        super(Discriminator, self).__init__()

        # Fully connected layer

        self.model = nn.Sequential(

            nn.Flatten(), # 4D Tensors => 1D
                
            nn.Linear(img_channels, 1024), # 100 => 256
            nn.LeakyReLU(),


            nn.Linear(1024, 512),
            nn.LeakyReLU(),

            nn.Linear(512, 256),
            nn.LeakyReLU(),

            nn.Linear(256, 1),
            nn.Sigmoid() # Probability of being fake or real

        )

    def forward(self, z):
        return self.model(img)


In [14]:
GAN_loss = nn.BCELoss() # Binary Cross Entropy loss

generator = Generator()
g_optimizer = optim.Adam(generator.parameter(), lr = 0.0002, betas = (0,5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameter(), lr = 0.0002, betas = (0,5, 0.999))



AttributeError: 'Generator' object has no attribute 'parameter'